In [1]:
import pandas as pd
import numpy as np

cab_rides = pd.read_csv("../data/cab_rides.csv")
weather = pd.read_csv("../data/weather.csv")

print(cab_rides.shape)
print(weather.shape)

(693071, 10)
(6276, 8)


In [2]:
cab_rides.head()

,distance,cab_type,time_stamp,destination,source,price,surge_multiplier,id,product_id,name
0,0.44,Lyft,1544952607890,North Station,Haymarket Square,5.0,1.0,424553bb-7174-41ea-aeb4-fe06d4f4b9d7,lyft_line,Shared
1,0.44,Lyft,1543284023677,North Station,Haymarket Square,11.0,1.0,4bd23055-6827-41c6-b23b-3c491f24e74d,lyft_premier,Lux
2,0.44,Lyft,1543366822198,North Station,Haymarket Square,7.0,1.0,981a3613-77af-4620-a42a-0c0866077d1e,lyft,Lyft
3,0.44,Lyft,1543553582749,North Station,Haymarket Square,26.0,1.0,c2d88af2-d278-4bfd-a8d0-29ca77cc5512,lyft_luxsuv,Lux Black XL
4,0.44,Lyft,1543463360223,North Station,Haymarket Square,9.0,1.0,e0126e1f-8ca9-4f2e-82b3-50505a09db9a,lyft_plus,Lyft XL


In [4]:
weather.head()

,temp,location,clouds,pressure,rain,time_stamp,humidity,wind
0,42.42,Back Bay,1.0,1012.14,0.1228,1545003901,0.77,11.25
1,42.43,Beacon Hill,1.0,1012.15,0.1846,1545003901,0.76,11.32
2,42.50,Boston University,1.0,1012.15,0.1089,1545003901,0.76,11.07
3,42.11,Fenway,1.0,1012.13,0.0969,1545003901,0.77,11.09
4,43.13,Financial District,1.0,1012.14,0.1786,1545003901,0.75,11.49


In [5]:
cab_rides.isnull().sum()

distance                0
cab_type                0
time_stamp              0
destination             0
source                  0
price               55095
surge_multiplier        0
id                      0
product_id              0
name                    0
dtype: int64

In [6]:
weather.isnull().sum()

temp             0
location         0
clouds           0
pressure         0
rain          5382
time_stamp       0
humidity         0
wind             0
dtype: int64

In [7]:
missing_price_rows = cab_rides[cab_rides["price"].isnull()]
missing_price_rows["name"].value_counts()

name
Taxi    55095
Name: count, dtype: int64

In [8]:
# every single row with a missing price belongs to the Taxi product, this product never had a price estimate captured during the scrape so there is nothing to fill in, dropping these rows is the correct call here
cab_rides = cab_rides[cab_rides["price"].notnull()].copy()
cab_rides.shape

(637976, 10)

In [9]:
cab_rides.duplicated().sum()

np.int64(0)

In [10]:
weather.duplicated().sum()

np.int64(0)

In [11]:
weather = weather.drop_duplicates().copy()
weather.shape

(6276, 8)

In [12]:
cab_rides["time_stamp"].head()

0    1544952607890
1    1543284023677
2    1543366822198
3    1543553582749
4    1543463360223
Name: time_stamp, dtype: int64

In [13]:
weather["time_stamp"].head()

0    1545003901
1    1545003901
2    1545003901
3    1545003901
4    1545003901
Name: time_stamp, dtype: int64

In [14]:
# the cab_rides timestamp is in milliseconds while the weather timestamp is in seconds, both need to be on the same unit before we can compare them, converting both to seconds and then to an actual datetime object
cab_rides["ride_time"] = pd.to_datetime(cab_rides["time_stamp"], unit="ms").astype("datetime64[ns]")
weather["weather_time"] = pd.to_datetime(weather["time_stamp"], unit="s").astype("datetime64[ns]")

In [15]:
print(cab_rides["ride_time"].min(), cab_rides["ride_time"].max())
print(weather["weather_time"].min(), weather["weather_time"].max())

2018-11-26 03:40:46.318000 2018-12-18 19:15:10.943000
2018-11-26 03:40:44 2018-12-18 18:45:02


In [16]:
sorted(cab_rides["source"].unique()) == sorted(weather["location"].unique())

True

In [17]:
# weather is only sampled roughly once an hour per location while a ride happens at any second, so this cannot be an exact match on time, for every ride we need to find the closest weather reading at the same source location and attach it to that ride, merge_asof does exactly this kind of nearest in time join but it needs both sides sorted by the time column first
cab_rides = cab_rides.sort_values("ride_time")
weather = weather.sort_values("weather_time")

In [18]:
cab_rides_with_weather = pd.merge_asof(
    cab_rides,
    weather,
    left_on="ride_time",
    right_on="weather_time",
    left_by="source",
    right_by="location",
    direction="nearest"
)
cab_rides_with_weather.shape

(637976, 20)

In [19]:
cab_rides_with_weather.head()

,distance,cab_type,time_stamp_x,destination,source,price,surge_multiplier,id,product_id,name,ride_time,temp,location,clouds,pressure,rain,time_stamp_y,humidity,wind,weather_time
0,3.03,Lyft,1543203646318,Theatre District,Boston University,34.0,1.0,ef4771c2-c88d-4730-aaf7-a95751e9d27e,lyft_luxsuv,Lux Black XL,2018-11-26 03:40:46.318,41.07,Boston University,0.86,1014.39,NaN,1543203645,0.92,1.36,2018-11-26 03:40:45
1,1.30,Uber,1543203646319,Theatre District,South Station,18.5,1.0,00ea74ea-2c49-416c-bfc5-f7877025f6eb,6c84fd89-3f11-4782-9b50-97c468b19529,Black,2018-11-26 03:40:46.319,40.86,South Station,0.87,1014.39,NaN,1543203644,0.93,1.60,2018-11-26 03:40:44
2,2.43,Lyft,1543203646320,Beacon Hill,Northeastern University,10.5,1.0,edfc7f44-97e1-48cd-930c-e4fe20e88ac8,lyft,Lyft,2018-11-26 03:40:46.320,40.81,Northeastern University,0.89,1014.35,NaN,1543203644,0.93,1.36,2018-11-26 03:40:44
3,2.71,Uber,1543203646320,Fenway,Theatre District,32.0,1.0,6172077a-22de-481b-aae2-b5763c87a6c4,6f72dfc5-27f1-42e8-84db-ccc7a75f6969,UberXL,2018-11-26 03:40:46.320,40.80,Theatre District,0.87,1014.39,NaN,1543203645,0.93,1.55,2018-11-26 03:40:45
4,2.71,Uber,1543203646320,Fenway,Theatre District,19.5,1.0,8682f9bf-5cc0-4dfc-b8fe-4e22070d1684,55c66225-fbe7-4fd5-9072-eab1ece5e23e,UberX,2018-11-26 03:40:46.320,40.80,Theatre District,0.87,1014.39,NaN,1543203645,0.93,1.55,2018-11-26 03:40:45


In [19]:
cab_rides_with_weather.isnull().sum()

distance                 0
cab_type                 0
time_stamp_x             0
destination              0
source                   0
price                    0
surge_multiplier         0
id                       0
product_id               0
name                     0
ride_time                0
temp                     0
location                 0
clouds                   0
pressure                 0
rain                544429
time_stamp_y             0
humidity                 0
wind                     0
weather_time             0
dtype: int64

In [20]:
# rain is missing whenever it simply did not rain at that location and time, the weather source only logs a row when there is measurable rain, so a missing rain value really means zero rain and not an unknown value
cab_rides_with_weather["rain"] = cab_rides_with_weather["rain"].fillna(0)

In [21]:
cab_rides_with_weather["time_gap_minutes"] = (cab_rides_with_weather["ride_time"] - cab_rides_with_weather["weather_time"]).abs().dt.total_seconds() / 60
cab_rides_with_weather["time_gap_minutes"].describe()

count    637976.000000
mean         12.256964
std           8.901646
min           0.000000
25%           4.966600
50%          10.127517
75%          19.894167
max          55.200983
Name: time_gap_minutes, dtype: float64

In [23]:
cab_rides_with_weather = cab_rides_with_weather.drop(columns=["time_stamp_x", "time_stamp_y", "location", "id", "product_id"])
cab_rides_with_weather.head()

KeyError: "['time_stamp_x', 'time_stamp_y', 'location', 'id', 'product_id'] not found in axis"

In [23]:
cab_rides_with_weather.to_csv("../data/cab_rides_with_weather.csv", index=False)